<a href="https://colab.research.google.com/github/hkbharath/LLM_Practice/blob/main/text_to_image_interactive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Reference: [Marktechpost](https://www.marktechpost.com/2025/02/19/steps-to-build-an-interactive-text-to-image-generation-application-using-gradio-and-hugging-faces-diffusers/?utm_source=www.airesearchinsights.com&utm_medium=newsletter&utm_campaign=featured-ais-google-ai-releases-gemma-3-and-hugging-face-releases-olympiccoder&_bhlid=df69e26b697c6ca191f8deb9493670353b5fbff5)

In [1]:
!pip install diffusers transformers accelerate gradio

In [2]:
import torch
from diffusers import StableDiffusionPipeline
import gradio as gr

# Global variable to cache the pipeline
pipe = None

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [3]:
torch.float16

torch.float16

In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [5]:
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
)
pipe = pipe.to("cuda")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
def generate_sd_image(prompt, num_inference_steps=50, guidance_scale=7.5):
    """
    Generate an image from a text prompt using Stable Diffusion.

    Args:
        prompt (str): Text prompt to guide image generation.
        num_inference_steps (int): Number of denoising steps (more steps can improve quality).
        guidance_scale (float): Controls how strongly the prompt is followed.

    Returns:
        PIL.Image: The generated image.
    """
    global pipe
    if pipe is None:
        print("Loading Stable Diffusion model... (this may take a while)")
        pipe = StableDiffusionPipeline.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            torch_dtype=torch.float16,
            revision="fp16"
        )
        pipe = pipe.to("cuda")

    # Use autocast for faster inference on GPU
    with torch.autocast("cuda"):
        image = pipe(prompt, num_inference_steps=num_inference_steps, guidance_scale=guidance_scale).images[0]

    return image

In [8]:
# Test with fewer steps
image = generate_sd_image("A software engineering playing guitar while wondering when he will get next interview call.", num_inference_steps=20, guidance_scale=7.5)

  0%|          | 0/20 [00:00<?, ?it/s]

In [9]:
# Define the Gradio interface
demo = gr.Interface(
    fn=generate_sd_image,
    inputs=[
        gr.Textbox(lines=2, placeholder="Enter your prompt here...", label="Text Prompt"),
        gr.Slider(minimum=10, maximum=100, step=5, value=50, label="Inference Steps"),
        gr.Slider(minimum=1, maximum=20, step=0.5, value=7.5, label="Guidance Scale")
    ],
    outputs=gr.Image(type="pil", label="Generated Image"),
    title="Stable Diffusion Text-to-Image Demo",
    description="Enter a text prompt to generate an image using Stable Diffusion. Adjust the parameters to fine-tune the result."
)

# Launch the interactive demo
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://617fac6fe5dbbe59bf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
